### RAG Pipelines-Data Ingestion to Vector DB Pipeline

In [7]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
# Read all the PDF in the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: SQL_1.pdf
  ✓ Loaded 2 pages

Processing: SQL_2.pdf
  ✓ Loaded 6 pages

Total documents loaded: 8


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20250507163210', 'source': '..\\data\\pdf\\SQL_1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SQL_1.pdf', 'file_type': 'pdf'}, page_content='1. What is SQL?\nSQL (Structured Query Language) is a standard language used to store, retrieve, and manipulate\ndata in relational databases.\n2. What is the difference between INNER JOIN and LEFT JOIN?\n- INNER JOIN returns only matching rows from both tables.\n- LEFT JOIN returns all rows from the left table, and matched rows from the right table (or NULLs if\nno match).\n3. What is the difference between ON and WHERE clause in SQL joins?\n- ON is used to define the join condition between two tables.\n- WHERE is used to filter the final result set after the join is performed.\n4. What is the difference between UNION and UNION ALL?\n- UNION removes duplicate rows.\n- UNION ALL keeps all rows, including dup

In [10]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [11]:
chunks=split_documents(all_pdf_documents)
chunks

Split 8 documents into 9 chunks

Example chunk:
Content: 1. What is SQL?
SQL (Structured Query Language) is a standard language used to store, retrieve, and manipulate
data in relational databases.
2. What is the difference between INNER JOIN and LEFT JOIN?...
Metadata: {'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20250507163210', 'source': '..\\data\\pdf\\SQL_1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SQL_1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20250507163210', 'source': '..\\data\\pdf\\SQL_1.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'SQL_1.pdf', 'file_type': 'pdf'}, page_content='1. What is SQL?\nSQL (Structured Query Language) is a standard language used to store, retrieve, and manipulate\ndata in relational databases.\n2. What is the difference between INNER JOIN and LEFT JOIN?\n- INNER JOIN returns only matching rows from both tables.\n- LEFT JOIN returns all rows from the left table, and matched rows from the right table (or NULLs if\nno match).\n3. What is the difference between ON and WHERE clause in SQL joins?\n- ON is used to define the join condition between two tables.\n- WHERE is used to filter the final result set after the join is performed.\n4. What is the difference between UNION and UNION ALL?\n- UNION removes duplicate rows.\n- UNION ALL keeps all rows, including dup

### Embedding and VectorStore DB

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity